# Limpieza de Datos y Análisis Exploratorio en Databricks (PySpark)
Notebook completo: limpieza + análisis exploratorio básico.


## CHUNK 1 — Cargar el CSV

In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/workspace/predictibilidad/semana6/dirty_cafe_sales (1).csv"

raw = (spark.read
       .option("header", "true")
       .csv(ruta))

display(raw.limit(10))
print("Columnas:", raw.columns)


## CHUNK 2 — Renombrar columnas

In [0]:
df = raw.select(
    F.col("Transaction ID").alias("id_transaccion"),
    F.col("Item").alias("producto"),
    F.col("Quantity").alias("cantidad"),
    F.col("Price Per Unit").alias("precio_unitario"),
    F.col("Total Spent").alias("monto_total"),
    F.col("Payment Method").alias("metodo_pago"),
    F.col("Location").alias("sucursal"),
    F.col("Transaction Date").alias("fecha_compra")
)
display(df.limit(10))


## CHUNK 3 — Limpieza de texto

In [0]:
def limpiar_texto(col):
    x = F.trim(F.col(col).cast("string"))
    x = F.regexp_replace(x, r"\s+", " ")
    return F.when(
        x.isNull() | (x == "") | (F.upper(x).isin("UNKNOWN", "ERROR", "NULL", "NONE", "NAN")),
        F.lit(None)
    ).otherwise(F.lower(x))

for c in ["id_transaccion", "producto", "metodo_pago", "sucursal", "fecha_compra"]:
    df = df.withColumn(c, limpiar_texto(c))
display(df.limit(10))


## CHUNK 4 — Convertir a número

In [0]:
def limpiar_texto(col):
    x = F.trim(F.col(col).cast("string"))
    x = F.regexp_replace(x, r"\s+", " ")
    return F.when(
        x.isNull() | (x == "") | (F.upper(x).isin("UNKNOWN", "ERROR", "NULL", "NONE", "NAN")),
        F.lit(None)
    ).otherwise(F.lower(x))

def a_numero(col):
    x = limpiar_texto(col)
    x = F.regexp_replace(x, ",", ".")
    x = F.regexp_replace(x, r"[^0-9\.\-]+", "")
    x = F.when((x == "") | x.isNull(), F.lit(None)).otherwise(x)
    return x.cast("double")

# Eliminar observaciones con 'ERROR' en monto_total antes de procesar
df = df.filter(~(F.upper(F.col("monto_total")) == "ERROR"))

for c in ["precio_unitario", "monto_total"]:
    df = df.withColumn(c, limpiar_texto(c))

df = (df
      .withColumn("cantidad", F.col("cantidad").cast("double"))
      .withColumn("precio_unitario", a_numero("precio_unitario"))
      .withColumn("monto_total", a_numero("monto_total"))
)
display(df.select("cantidad", "precio_unitario", "monto_total").limit(10))

## CHUNK 5 — Convertir a fecha

In [0]:
df = df.withColumn(
    "fecha_compra",
    F.coalesce(
        F.to_timestamp("fecha_compra", "MM/dd/yyyy"),
        F.to_timestamp("fecha_compra", "yyyy-MM-dd"),
        F.to_timestamp("fecha_compra", "dd/MM/yyyy"),
        F.to_timestamp("fecha_compra")
    )
)
display(df.select("fecha_compra").limit(10))


## CHUNK 6 — Quitar duplicados

In [0]:
df = df.distinct()
print("Filas después de quitar duplicados:", df.count())


## CHUNK 7 — Imputación

In [0]:
med_monto  = df.selectExpr("percentile_approx(monto_total, 0.5, 10000) as med").collect()[0]["med"]
med_cant   = df.selectExpr("percentile_approx(cantidad, 0.5, 10000) as med").collect()[0]["med"]
med_precio = df.selectExpr("percentile_approx(precio_unitario, 0.5, 10000) as med").collect()[0]["med"]

df = (df
      .withColumn("monto_total", F.when(F.col("monto_total").isNull(), F.lit(med_monto)).otherwise(F.col("monto_total")))
      .withColumn("cantidad", F.when(F.col("cantidad").isNull(), F.round(F.lit(med_cant))).otherwise(F.col("cantidad")))
      .withColumn("precio_unitario", F.when(F.col("precio_unitario").isNull(), F.lit(med_precio)).otherwise(F.col("precio_unitario")))
)

df = df.filter(F.col("fecha_compra").isNotNull())
df = df.filter(F.col("id_transaccion").isNotNull())

df = (df
      .withColumn("producto", F.when(F.col("producto").isNull(), F.lit("desconocido")).otherwise(F.col("producto")))
      .withColumn("metodo_pago", F.when(F.col("metodo_pago").isNull(), F.lit("no_definido")).otherwise(F.col("metodo_pago")))
      .withColumn("sucursal", F.when(F.col("sucursal").isNull(), F.lit("sin_sucursal")).otherwise(F.col("sucursal")))
)
display(df.limit(10))


# Análisis Exploratorio de Datos (EDA)

## 1. Estadísticas descriptivas generales

In [0]:
display(df.select("cantidad", "precio_unitario", "monto_total").describe())


## 2. Ventas totales por producto

In [0]:
ventas_producto = (df.groupBy("producto")
                     .agg(F.sum("monto_total").alias("ventas_totales"))
                     .orderBy(F.desc("ventas_totales")))
display(ventas_producto)


## 3. Cantidad de transacciones por método de pago

In [0]:
display(df.groupBy("metodo_pago").count().orderBy(F.desc("count")))


## 4. Ventas por sucursal

In [0]:
ventas_sucursal = (df.groupBy("sucursal")
                      .agg(F.sum("monto_total").alias("ventas_totales"))
                      .orderBy(F.desc("ventas_totales")))
display(ventas_sucursal)


## 5. Evolución temporal de ventas

In [0]:
ventas_fecha = (df.groupBy(F.to_date("fecha_compra").alias("fecha"))
                   .agg(F.sum("monto_total").alias("ventas_totales"))
                   .orderBy("fecha"))
display(ventas_fecha)
